# [Lab 3](https://github.com/hanggrian/IIT-CS587/blob/assets/assignments/lab3.pdf): AutoGen Chat

> After you complete the tasks listed in the instructions document (`Lab_3 - Instructions-2.pdf`), submit (a link in this class discussion where anyone of your classmates will be able to watch) the Panopto Video Recording of a successful run on your computer.
>
> Update this file (OAI_CONFIG_LIST Download OAI_CONFIG_LIST) to add your OpenAI
  KEY and use gpt-4o-mini model.
>
> Run `agentchat_nestedchat.ipynb`.
>
> Run ` agentchat_nested_sequential_chats.ipynb`.
>
> For this lab you have been asked to install/run AutoGen nested-chats scripts on your computer:

[Screen recording](https://github.com/hanggrian/IIT-CS587/raw/assets/lab3/screenrecord.mp4)

## Requirements

1.  [x] Install Python version 3.10: [python310<sup>AUR</sup>](https://aur.archlinux.org/packages/python310)
1.  [x] Install development environment:
    - VSCode: [visual-studio-code-bin<sup>AUR</sup>](https://aur.archlinux.org/packages/visual-studio-code-bin)
    - Jupyter: [jupyterlab](https://archlinux.org/packages/extra/any/jupyterlab)
1.  [x] Check Python version.
    ```sh
    python --version
    ```
1.  [x] Create a virtual environment and link notebook kernel.
    ```sh
    source .venv/bin/activate
    python -m ipykernel install --user --name=.venv
    ```
1.  [x] Install packages:
    - `uv` project manager.
      ```sh
      pip install uv
      ```
    - `pyautogen` and dependencies for lessons.
      ```
      uv pip install -r requirements.txt
      ```
1.  [x] Create an OpenAI account and API key.
1.  [x] Set up the environment variable.
    ```sh
    echo 'OPENAI_API_KEY=XXXX-XXXX' >> ~/.env
    ```
1.  [x] Run [Solving Complex Tasks with A Sequence of Nested Chats](https://github.com/ag2ai/ag2/blob/main/notebook/agentchat_nested_sequential_chats.ipynb).
1.  [x] Run [Solving Complex Tasks with Nested Chats](https://github.com/ag2ai/ag2/blob/main/notebook/agentchat_nestedchat.ipynb)

## Questions

> Did you encounter any issue part of the setup process?

No, the setup process was straightforward.

> What was the computer hardware (processor and main memory) that you have on your computer?

- **CPU:** Intel Core i5-10400
- **RAM:** 32GB DDR4
- 
> How long did it take you to complete the installation?

This portion did not require any installation, because the required packages are similar to Lab 1.

> How long did it take AutoGen to complete the entire conversation of the AI Agents?

It took two minutes to complete each notebook, so four minutes in total.

> Comment on the quality of the output produced.

The first notebook has a simpler code structure and took less time to complete. There was a Pydantic warning towards the end of second notebook execution, but the output was still generated successfully.

## Setup

In [3]:
from utils import get_openai_api_key

llm_config = {
    'model': 'gpt-4-turbo',
    'api_key': get_openai_api_key(),
}

## A sequence of nested chats

> Suppose we want the agents to complete the following sequence of tasks. Since the first task could be complex to solve, lets construct new agents that can serve as an inner monologue.

In [4]:
tasks = [
    'On which days in 2024 was Microsoft Stock higher than $400? Comment on the stock performance.',
    'Make a pleasant joke about it.',
]

#### Step 1: Define agents

> ##### A Group Chat for Inner Monologue
>
> Below, we construct a group chat manager which manages an inner_assistant agent and an inner_code_interpreter agent. Later we will use this group chat inside another agent.
>
> ##### Inner- and Outer-Level Individual Agents
>
> Now we will construct a number of individual agents that will assume role of outer and inner agents.

In [ ]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager

inner_assistant = \
    AssistantAgent(
        'Inner-assistant',
        llm_config=llm_config,
        is_termination_msg=lambda x: x.get('content', '').find('TERMINATE') >= 0,
    )
inner_code_interpreter = \
    UserProxyAgent(
        'Inner-code-interpreter',
        human_input_mode='NEVER',
        code_execution_config={
            'work_dir': 'coding',
            'use_docker': False,
        },
        default_auto_reply='',
        is_termination_msg=lambda x: x.get('content', '').find('TERMINATE') >= 0,
    )
groupchat = \
    GroupChat(
        agents=[inner_assistant, inner_code_interpreter],
        messages=[],
        speaker_selection_method='round_robin',
        allow_repeat_speaker=False,
        max_round=8,
    )
manager = \
    GroupChatManager(
        groupchat=groupchat,
        is_termination_msg=lambda x: x.get('content', '').find('TERMINATE') >= 0,
        llm_config=llm_config,
        code_execution_config={
            'work_dir': 'coding',
            'use_docker': False,
        },
    )

In [6]:
from textwrap import dedent

assistant_1 = \
    AssistantAgent(
        name='Assistant_1',
        llm_config=llm_config,
    )
assistant_2 = \
    AssistantAgent(
        name='Assistant_2',
        llm_config=llm_config,
    )
writer = \
    AssistantAgent(
        name='Writer',
        llm_config=llm_config,
        system_message= \
            dedent(
                '''\
                You are a professional writer, known for
                your insightful and engaging articles.
                You transform complex concepts into compelling narratives.''',
            ),
    )
reviewer = \
    AssistantAgent(
        name='Reviewer',
        llm_config=llm_config,
        system_message= \
            dedent(
                '''\
                You are a compliance reviewer, known for your thoroughness and commitment to standards.
                Your task is to scrutinize content for any harmful elements or regulatory violations, ensuring
                all materials align with required guidelines.
                You must review carefully, identify potential issues, and maintain the integrity of the organization.
                Your role demands fairness, a deep understanding of regulations, and a focus on protecting against
                harm while upholding a culture of responsibility.''',
            ),
    )
user = \
    UserProxyAgent(
        name='User',
        human_input_mode='NEVER',
        is_termination_msg=lambda x: x.get('content', '').find('TERMINATE') >= 0,
        code_execution_config={
            'last_n_messages': 1,
            'work_dir': 'tasks',
            'use_docker': False,
        },
    )

#### Step 2: Orchestrate nested chats to solve tasks

> ##### Outer level
>
> In the following code block, at the outer level, we have communication between:
>
> - `user` - `assistant_1` for solving the first task, i.e., `tasks[0]`.
> - `user` - `assistant_2` for solving the second task, i.e., `tasks[1]`.
>
> ##### Inner level (nested chats)
>
> Since the first task is quite complicated, we created a sequence of nested chats as the inner monologue of Assistant_1.
>
> 1.  `assistant_1` - `manager`: This chat intends to delegate the task received by Assistant_1 to the Manager to solve.
> 1.  `assistant_1` - `writer`: This chat takes the output from Nested Chat 1, i.e., Assistant_1 vs. Manager, and lets the Writer polish the content to make an engaging and nicely formatted blog post, which is realized through the writing_message function.
> 1.  `assistant_1` - `reviewer`: This chat takes the output from Nested Chat 2 and intends to let the Reviewer agent review the content from Nested Chat 2.
> 1.  `assistant_1` - `writer`: This chat takes the output from previous nested chats and intends to let the Writer agent finalize a blog post.
>
> The sequence of nested chats can be realized with the `register_nested_chats` function, which allows one to register one or a sequence of chats to a particular agent (in this example, the `assistant_1` agent).
>
> Information about the sequence of chats can be specified in the `chat_queue` argument of the `register_nested_chats` function. The following fields are especially useful:
>
> - `recipient` (required) specifies the nested agent;
> - `message` specifies what message to send to the nested recipient agent. In a sequence of nested chats, if the `message` field is not specified, we will use the last message the registering agent received as the initial message in the first chat and will skip any subsequent chat in the queue that does not have the `message` field. You can either provide a string or define a callable that returns a string.
> - `summary_method` decides what to get out of the nested chat. You can either select from existing options including `'last_msg'` and `'reflection_with_llm'`, or or define your own way on what to get from the nested chat with a Callable.
> - `max_turns` determines how many turns of conversation to have between the concerned agent pairs.

In [7]:
def writing_message(recipient, messages, sender, config):
    return 'Polish the content to make an engaging and nicely formatted blog post. \n\n ' + \
        f"{recipient.chat_messages_for_summary(sender)[-1]['content']}"

nested_chat_queue = [
    {
        'recipient': manager,
        'summary_method': 'reflection_with_llm',
        'clear_history': False,
    },
    {
        'recipient': writer,
        'message': writing_message,
        'summary_method': 'last_msg',
        'max_turns': 1,
    },
    {
        'recipient': reviewer,
        'message': 'Review the content provided.',
        'summary_method': 'last_msg',
        'max_turns': 1,
    },
    {
        'recipient': writer,
        'message': writing_message,
        'summary_method': 'last_msg',
        'max_turns': 1,
    },
]
assistant_1.register_nested_chats(nested_chat_queue,trigger=user)
res = \
    user.initiate_chats([
        {
            'recipient': assistant_1,
            'message': tasks[0],
            'max_turns': 1,
            'summary_method': 'last_msg',
        },
        {
            'recipient': assistant_2,
            'message': tasks[1],
        },
    ])


********************************************************************************
Starting a new chat....

********************************************************************************
User (to Assistant_1):

On which days in 2024 was Microsoft Stock higher than $400? Comment on the stock performance.

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
Assistant_1 (to chat_manager):

On which days in 2024 was Microsoft Stock higher than $400? Comment on the stock performance.

--------------------------------------------------------------------------------

Next speaker: Inner-assistant

Inner-assistant (to chat_manager):

To gather the data on Microsoft's stock prices for the year 2024 and check which days the stock price was higher than $400, usually, we would downlo

/home/hanggrian/GitHub/IIT-CS587/proj/.venv/lib/python3.10/site-packages/autogen/agentchat/chat.py:47: UserWarning: Repetitive recipients detected: The chat history will be cleared by default if a recipient appears more than once. To retain the chat history, please set 'clear_history=False' in the configuration of the repeating agent.
  warnings.warn(


## Nested chats

> Suppose we want the agents to complete the following sequence of tasks:

In [8]:
task = 'Write a concise but engaging blogpost about Nvida.'

### Scenario 1

> Let's say we desire the following workflow to solve the task: a user_proxy agent issues the initial query to a writer and acts as a proxy for the user. Whenever an initial writing is provided, a critic should be invoked to offer critique as feedback. This workflow can be realized by a three-agent system shown below. The system includes a user_proxy agent and a writer agent communicating with each other, with a critic agent nested within the user_proxy agent to provide critique. Whenever the user_proxy receives a message from the writer, it engages in a conversation with the critic agent to work out feedback on the writer's message.

#### Step 1: Define Agents

> Define the agents, including the outer agents writer and user_proxy, and the inner agent critic.

In [9]:
writer = \
    AssistantAgent(
        name='Writer',
        llm_config=llm_config,
        system_message= \
            dedent(
                '''\
                You are a professional writer, known for your insightful and engaging articles.
                You transform complex concepts into compelling narratives.
                You should improve the quality of the content based on the feedback from the user.''',
            ),
    )
user_proxy = \
    UserProxyAgent(
        name='User',
        human_input_mode='NEVER',
        is_termination_msg=lambda x: x.get('content', '').find('TERMINATE') >= 0,
        code_execution_config={
            'last_n_messages': 1,
            'work_dir': 'tasks',
            'use_docker': False,
        },
    )
critic = \
    AssistantAgent(
        name='Critic',
        llm_config=llm_config,
        system_message= \
            dedent(
                '''\
                You are a critic, known for your thoroughness and commitment to standards.
                Your task is to scrutinize content for any harmful elements or regulatory violations, ensuring
                all materials align with required guidelines.
                For code''',
            ),
    )

#### Step 2: Orchestrate Nested Chats to Solve Tasks

In [10]:
def reflection_message(recipient, messages, sender, config):
    print('Reflecting...', 'yellow')
    return 'Reflect and provide critique on the following writing. \n\n ' + \
        f"{recipient.chat_messages_for_summary(sender)[-1]['content']}"

user_proxy.register_nested_chats(
    [
        {
            'recipient': critic,
            'message': reflection_message,
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
    ],
    trigger=writer,
)
res = user_proxy.initiate_chat(recipient=writer, message=task, max_turns=2, summary_method='last_msg')

User (to Writer):

Write a concise but engaging blogpost about Nvida.

--------------------------------------------------------------------------------
Writer (to User):

## Unveiling Nvidia: Pioneers of the Visual Computing Frontier

In the high-powered world of tech innovation, few companies shine as brightly as Nvidia. Founded in 1993, Nvidia has transcended its humble beginnings to become a titan in the realm of visual computing, forever altering how we interact with digital technologies.

### The GPU Revolution

At the heart of Nvidia's innovation is the Graphics Processing Unit (GPU), a breakthrough that catapulted the company into the limelight. Originally designed to enhance the gaming experience with stunning visuals and seamless gameplay, Nvidia's GPUs have far surpassed these initial applications. Today, they are indispensable tools for a vast range of industries, from film production to artificial intelligence (AI).

### Beyond Gaming: Nvidia's Diverse Landscape

While Nvid

### Scenarios 2

> Let's say we desire the following workflow to solve the task. Compared to scenario 1, we want to include an additional `critic_executor` agent to chat with the critic and execute some tool calls involved in the chat. For example, a tool for detecting harmful content in the output of the writer.
>
> This workflow can be realized by a four-agent system shown below. The system includes a user_proxy agent and a writer agent communicating with each other, with a chat between the `critic` and `critic_executor` agent nested within the `user_proxy` agent to provide critique. Whenever the user_proxy receives a message from the writer, it engages in a conversation between `critic` and `critic_executor` to work out feedback on the writer's message. A summary of the nested conversation will be passed to the user_proxy, which will then be passed to the writer as feedback.

In [11]:
from typing import Annotated

critic_executor = \
    UserProxyAgent(
        name='Critic_Executor',
        human_input_mode='NEVER',
        code_execution_config={
            'last_n_messages': 1,
            'work_dir': 'tasks',
            'use_docker': False,
        },
    )

@critic_executor.register_for_execution()
@critic.register_for_llm(name='check_harmful_content', description='Check if content contain harmful keywords.')
def check_harmful_content(content: Annotated[str, 'Content to check if harmful keywords.']):
    harmful_keywords = ['violence', 'hate', 'bullying', 'death']
    text = content.lower()
    print(f'Checking for harmful content...{text}', 'yellow')
    for keyword in harmful_keywords:
        if keyword in text:
            return f'Denied. Harmful content detected:{keyword}'
    return 'Approve. TERMINATE'

def reflection_message_no_harm(recipient, messages, sender, config):
    print('Reflecting...', 'yellow')
    return 'Reflect and provide critique on the following writing. ' + \
        'Ensure it does not contain harmful content. You can use tools to check it. \n\n ' + \
        f"{recipient.chat_messages_for_summary(sender)[-1]['content']}"

user_proxy.register_nested_chats(
    [
        {
            'sender': critic_executor,
            'recipient': critic,
            'message': reflection_message_no_harm,
            'max_turns': 2,
            'summary_method': 'last_msg',
        }
    ],
    trigger=writer,
)
res = user_proxy.initiate_chat(recipient=writer, message=task, max_turns=2, summary_method='last_msg')

The return type of the function 'check_harmful_content' is not annotated. Although annotating it is optional, the function should return either a string, a subclass of 'pydantic.BaseModel'.


User (to Writer):

Write a concise but engaging blogpost about Nvida.

--------------------------------------------------------------------------------
Writer (to User):

## Unveiling Nvidia: Pioneers of the Visual Computing Frontier

In the high-powered world of tech innovation, few companies shine as brightly as Nvidia. Founded in 1993, Nvidia has transcended its humble beginnings to become a titan in the realm of visual computing, forever altering how we interact with digital technologies.

### The GPU Revolution

At the heart of Nvidia's innovation is the Graphics Processing Unit (GPU), a breakthrough that catapulted the company into the limelight. Originally designed to enhance the gaming experience with stunning visuals and seamless gameplay, Nvidia's GPUs have far surpassed these initial applications. Today, they are indispensable tools for a vast range of industries, from film production to artificial intelligence (AI).

### Beyond Gaming: Nvidia's Diverse Landscape

While Nvid